In [1]:
import numpy as np
import scanpy as sc
import plotly.graph_objects as go
import scipy.sparse as sp
import pandas as pd
import plotly.express as px


In [2]:
# File paths
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/20230918_3xTgAD_oldfemale_1_pos_3_1000.h5ad"

# Load and process
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/20230918_3xTgAD_oldfemale_1_pos_3_1000.h5ad


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [3]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'
    var: 'mzs'
    uns: 'spatial'
    obsm: 'spatial'
    layers: 'tic_normalized'


In [4]:
adata.var

,mzs
273.0411,273.0411
137.0243,137.0243
274.0449,274.0449
331.0315,331.0315
369.3527,369.3527
...,...
558.9552,558.9552
537.9756,537.9756
340.0839,340.0839
533.0795,533.0795


In [5]:
adata.X

<16605x1000 sparse matrix of type '<class 'numpy.float64'>'
	with 14363629 stored elements in Compressed Sparse Row format>

In [6]:
def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

In [7]:
X = adata.X
X = X.toarray() if hasattr(X, "toarray") else X
mz_axis, global_spec = create_global_spectrum(X, adata.var_names.astype(float).values)

mz_peaks = mz_axis
intensities = global_spec

In [8]:
def interactive_stem_spectrum_pixel(mz_axis, intensities, mz_min=None, mz_max=None):

    # Filter by mz range if provided
    if mz_min is not None and mz_max is not None:
        mask = (mz_axis >= mz_min) & (mz_axis <= mz_max)
        mz_axis = mz_axis[mask]
        spectrum = intensities[mask]

    # Create the stem lines (from 0 to intensity)
    fig = go.Figure()

    for mz, intensity in zip(mz_axis, spectrum):
        fig.add_trace(go.Scatter(
            x=[mz, mz],
            y=[0, intensity],
            mode="lines",
            line=dict(color="blue", width=1),
            hoverinfo="skip"
        ))

    # Add the tips of the stems
    fig.add_trace(go.Scatter(
        x=mz_axis,
        y=spectrum,
        mode='none',
        hovertemplate='m/z: %{x:.4f}<br>Intensity: %{y:.2f}',
        name='Peaks'
    ))

    fig.update_layout(
        title=f"Interactive Stem Plot",
        xaxis_title="m/z",
        yaxis_title="Intensity",
        template="plotly_white",
        height=400,
        hovermode="closest"
    )

    fig.show()

In [9]:
interactive_stem_spectrum_pixel(mz_peaks, intensities, mz_min=0, mz_max=2000)

In [10]:
def plot_mz_across_sample_custom(adata, target_mz, tol=0.1):
    """
    Plots the intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Extract spatial coordinates (assumed to be in adata.obs["x"] and adata.obs["y"])
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    import pandas as pd
    df = pd.DataFrame({"x": x, "y": y, "intensity": intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale = [
    [0.0, "#000000"],  # black
    [0.10, "#000080"],  # navy
    [0.15, "#0000FF"],  # blue
    [0.30, "#8000FF"],  # purple-ish
    [0.45, "#FF0000"],  # red
    [0.60, "#FF8000"],  # orange
    [0.75, "#FFFF00"],  # yellow
    [1.0, "#FFFFFF"]   # white
],
        title=f"Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [11]:
import numpy as np
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs("images", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:1000]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)

# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_hdimaging/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_hdimaging/mz_273.0411.png
Saved: images_hdimaging/mz_137.0243.png
Saved: images_hdimaging/mz_274.0449.png
Saved: images_hdimaging/mz_369.3527.png
Saved: images_hdimaging/mz_195.0927.png
Saved: images_hdimaging/mz_331.0315.png
Saved: images_hdimaging/mz_257.0457.png
Saved: images_hdimaging/mz_313.0321.png
Saved: images_hdimaging/mz_637.3065.png
Saved: images_hdimaging/mz_275.0543.png
Saved: images_hdimaging/mz_245.0452.png
Saved: images_hdimaging/mz_798.5422.png
Saved: images_hdimaging/mz_197.9910.png
Saved: images_hdimaging/mz_760.5860.png
Saved: images_hdimaging/mz_244.0375.png
Saved: images_hdimaging/mz_246.0524.png
Saved: images_hdimaging/mz_333.0229.png
Saved: images_hdimaging/mz_347.0090.png
Saved: images_hdimaging/mz_258.0513.png
Saved: images_hdimaging/mz_772.5265.png
Saved: images_hdimaging/mz_229.0501.png
Saved: images_hdimaging/mz_330.0238.png
Saved: images_hdimaging/mz_734.5704.png
Saved: images_hdimaging/mz_198.9986.png
Saved: images_hdimaging/mz_332.0322.png


In [12]:
def normalize_by_tic(adata, layer_name="tic_normalized"):
    # Extract intensity matrix
    X = adata.X
    is_sparse = sp.issparse(X)

    # Compute TIC for each pixel
    tic = X.sum(axis=1)
    if is_sparse:
        tic = np.array(tic).flatten()
    else:
        tic = tic.flatten()

    # Prevent division by zero
    tic[tic == 0] = np.nan

    # Store TIC in obs
    adata.obs["TIC"] = tic

    # Normalize each pixel spectrum by its TIC
    if is_sparse:
        from scipy.sparse import diags
        D_inv = diags(1.0 / tic)
        X_norm = D_inv @ X
    else:
        X_norm = X / tic[:, np.newaxis]

    # Replace NaNs with zeros
    if sp.issparse(X_norm):
        X_norm.data[np.isnan(X_norm.data)] = 0
    else:
        X_norm = np.nan_to_num(X_norm)

    # Save as new layer
    adata.layers[layer_name] = X_norm
    return adata

In [13]:
# Ensure the image folder exists
os.makedirs("images", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:1000]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)
adata = normalize_by_tic(adata, layer_name="tic_normalized")

In [14]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.layers["tic_normalized"][:, mz_index].toarray().flatten() if hasattr(adata.layers["tic_normalized"], "toarray") else adata.layers["tic_normalized"][:, mz_index]

    # Extract spatial coordinates (assumed to be in adata.obs["x"] and adata.obs["y"])
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    import pandas as pd
    df = pd.DataFrame({"x": x, "y": y, "intensity": intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale = [
    [0.0, "#000000"],  # black
    [0.10, "#000080"],  # navy
    [0.15, "#0000FF"],  # blue
    [0.30, "#8000FF"],  # purple-ish
    [0.45, "#FF0000"],  # red
    [0.60, "#FF8000"],  # orange
    [0.75, "#FFFF00"],  # yellow
    [1.0, "#FFFFFF"]   # white
    ],
        title=f"TIC Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [15]:
import numpy as np
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs("images", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:1000]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)

# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_hdimaging_tic/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_hdimaging_tic/mz_273.0411.png
Saved: images_hdimaging_tic/mz_137.0243.png
Saved: images_hdimaging_tic/mz_274.0449.png
Saved: images_hdimaging_tic/mz_369.3527.png
Saved: images_hdimaging_tic/mz_195.0927.png
Saved: images_hdimaging_tic/mz_331.0315.png
Saved: images_hdimaging_tic/mz_257.0457.png
Saved: images_hdimaging_tic/mz_313.0321.png
Saved: images_hdimaging_tic/mz_637.3065.png
Saved: images_hdimaging_tic/mz_275.0543.png
Saved: images_hdimaging_tic/mz_245.0452.png
Saved: images_hdimaging_tic/mz_798.5422.png
Saved: images_hdimaging_tic/mz_197.9910.png
Saved: images_hdimaging_tic/mz_760.5860.png
Saved: images_hdimaging_tic/mz_244.0375.png
Saved: images_hdimaging_tic/mz_246.0524.png
Saved: images_hdimaging_tic/mz_333.0229.png
Saved: images_hdimaging_tic/mz_347.0090.png
Saved: images_hdimaging_tic/mz_258.0513.png
Saved: images_hdimaging_tic/mz_772.5265.png
Saved: images_hdimaging_tic/mz_229.0501.png
Saved: images_hdimaging_tic/mz_330.0238.png
Saved: images_hdimaging_tic/mz_7